```{toctree}
:maxdepth: 2
:caption: Contents:

# Walkthrough Example

This notebook provides a practical walkthrough of the `gis` subpackage of `oceanicospy`. It covers reading, writing, reprojecting, merging and masking XYZ point cloud data for oceanographic and coastal applications. All examples follow a single dataset: a topobathymetric survey of a coastal area, composed of a high-resolution bathymetric tile (10 m, EPSG:9377) and a lower-resolution tile (50 m, EPSG:32617) stored as a point shapefile. Together they illustrate a complete preprocessing pipeline from raw input files to a merged, analysis-ready point cloud.

```{hint}
To replicate the examples in this notebook you will need:
- `oceanicospy` installed in your Python environment.
- Two point data files in your working directory:
  - A headerless XYZ text file.
  - A point shapefile (.shp).

For the merging section to be meaningful, the two files should cover overlapping areas. If they do not overlap, the merge will simply concatenate the data without resolving any conflicts.

Replace the file paths in the cells below with the paths to your own files. The workflow generalizes to any XYZ or point vector file.
```

## Importing libraries

The following libraries are imported to support the examples in this notebook.

In [ ]:
import pandas as pd
import geopandas as gpd

The different modules in the `oceanicospy.gis` subpackage are imported to showcase the gis capabilities.

In [ ]:
from oceanicospy.gis import (
    XYZFormatSpec,
    PointFileIO,
    PointFileReprojector,
    reproject_points,
    XYZMerger,
    AxisAlignedRectangle,
    XYZRectangleMask,
    ProfileAxis,
    ProfileInterpolator,
    Grid
)

## Point file I/O

The `gis` subpackage provides the `PointFileIO` class to read and write point data from XYZ and vector files. An `XYZFormatSpec` object describes the on-disk layout of XYZ files: column delimiter, whether a header row is present, and column names for X, Y and Z.

`XYZFormatSpec` describes how an XYZ file is structured on disk. When the format is known in advance, it is good practice to define it explicitly rather than relying on automatic detection. This ensures consistent behaviour across reading and writing operations.

In [ ]:
spec = XYZFormatSpec(
    delimiter=" ",
    has_header=False,
    x_column="x",
    y_column="y",
    z_column="z",
)
spec

When the file format is not known in advance, `PointFileIO` detects it automatically from the file content. The detected spec is accessible via the `format_spec` attribute.

In [ ]:
xyz_path = "../data/gis/raw/bat_sai_10m_9377.xyz"

reader = PointFileIO(xyz_path)
reader.format_spec

`PointFileIO.read` loads the file into a `pandas.DataFrame` using the detected or provided `format_spec`.

In [ ]:
df = reader.read()
df.head()

A quick inspection of the Z column shows the value range. In this example the dataset contains bathymetry only, so negative Z values correspond to points that fall on land and are excluded. If your dataset is topobathymetric, negative values represent land elevation and should be kept.

In [ ]:
df["z"].describe()

In [ ]:
df_bathy = df[df["z"] >= 0].reset_index(drop=True)
df_bathy["z"].describe()

`PointFileIO.write` saves a `pandas.DataFrame` to an XYZ plain-text file using the `format_spec` and path set at construction.

In [ ]:
bathy_path = "../data/gis/processed/bat_sai_10m_9377-filt.xyz"

writer = PointFileIO(bathy_path)
writer.write(df_bathy)

`PointFileIO.read_as_geodataframe` reads the file and returns a `geopandas.GeoDataFrame` with Point geometries. Unlike a plain DataFrame, a GeoDataFrame stores each point as a spatial object with an assigned CRS, which enables coordinate reprojection, spatial queries and export to vector formats such as `.shp` or `.geojson`. The CRS must be provided at construction since XYZ files carry no spatial metadata.

In [ ]:
reader_geo = PointFileIO(bathy_path, crs=9377)
gdf = reader_geo.read_as_geodataframe()
gdf.head()

## Reprojecting point files

The `crs` module provides tools to reproject point data between coordinate reference systems. Both vector files and XYZ text files are supported via `PointFileReprojector`. All EPSG codes can be passed as integers (e.g. `9377`) or strings (e.g. `"EPSG:9377"`).

### From a vector file

`PointFileReprojector` loads a point file and reads its CRS automatically from the embedded `.prj` file for vector formats. If the file has no embedded CRS, the source CRS can be specified manually via `source_epsg`. All file I/O is handled internally by `PointFileIO`. The data is then reprojected to the target CRS and exported as an XYZ file. 

Before loading, it is recommended to inspect the file column names to confirm the correct `z_column` name, as it may vary depending on the source software.

In [ ]:
shp_path = "../data/gis/raw/bat_sai_50m_32617.shp"
preview = PointFileIO(shp_path)
df_preview = preview.read()
print(df_preview.columns.tolist())
print(preview.read_as_geodataframe().crs)

In [ ]:
reprojector = PointFileReprojector(
    input_path=shp_path,
    z_column="field_3",
)
print(reprojector.crs)

In [ ]:
reprojector.reproject_to_epsg(9377)
print(reprojector.crs)

Once reprojected, the result is exported as an XYZ file using `to_xyz`, which internally delegates to `PointFileIO.write_from_geodataframe` from
the `point_io` module. The output is a plain-text file.

In [ ]:
vector_reprojected_path = "../data/gis/processed/bat_sai_50m_9377_from_vector.xyz"
reprojector.to_xyz(vector_reprojected_path)

For cases where inspecting or modifying the data between loading and exporting is not needed, `reproject_points` performs the full workflow in a single call.

In [ ]:
reproject_points(
    input_path=shp_path,
    output_path=vector_reprojected_path,
    target_epsg=9377,
    z_column="field_3",
)

### From an XYZ file

`reproject_xyz_file` is a convenience function that performs the full workflow in a single call: load, reproject and export. It is equivalent to using `PointFileReprojector` directly but without the intermediate steps. Use it when inspecting or modifying the data between loading and exporting is not needed.

In [ ]:
xyz_input_path = "../data/gis/raw/bat_sai_50m_32617.xyz"
xyz_output_path = "../data/gis/processed/bat_sai_50m_9377_from_xyz.xyz"

reproject_points(
    input_path=xyz_input_path,
    output_path=xyz_output_path,
    target_epsg=9377,
    source_epsg=32617,
)

## Merging XYZ tiles

`XYZMerger` combines multiple XYZ tiles into a single point dataset. Overlapping regions are resolved by priority: points from lower-priority tiles that fall within the coverage footprint of any higher-priority tile are discarded. All input tiles must share the same CRS and on-disk format before merging. Use `PointFileReprojector` and `PointFileIO` to standardize them beforehand if needed. The priority order is declared explicitly, the first filename in the list has the highest priority.

In [ ]:
tiles_dir = "../data/gis/processed"
merged_path = "../data/gis/processed/bat_sai_10m-50m_9377_merged.xyz"

priority = [
    "bat_sai_10m_9377-filt.xyz",   # highest priority
    "bat_sai_50m_9377_from_vector.xyz",    # lowest priority
]

merger = XYZMerger(
    input_dir=tiles_dir,
    priority=priority,
    crs=9377,
)

`run_merge` executes the full pipeline and returns the merged DataFrame, which can be inspected directly without reading the output file.

In [ ]:
df_merged = merger.run_merge(merged_path)
df_merged.head()

By default, `XYZMerger` uses internally calibrated values for the coverage polygon construction. If the merge result shows lower-priority points leaking into high-priority areas, or too many valid points being discarded, the `buffer_factor` and `closing_factor` parameters can be adjusted directly in the `XYZMerger` constructor.

## Masking XYZ data

`XYZRectangleMask` filters an XYZ dataset by one or more axis-aligned rectangular regions. Two modes are available: `"keep"` retains only
points inside the rectangles, and `"exclude"` removes them. Rectangles are defined by two opposite corners in the same coordinate space as the data.

In [ ]:
x1 = 4051000
y1 = 2961500
x2 = 4058000
y2 = 2967000
rect1 = AxisAlignedRectangle(p1=(x1, y1), p2=(x2, y2))

Multiple rectangles can be combined in a single mask. All rectangles are evaluated simultaneously, a point is considered inside if it falls within any of them.

In [ ]:
masker = XYZRectangleMask(rectangles=[rect1], mode="keep")
df_masked = masker.filter_dataframe(df_merged)

print(f"Points before masking: {len(df_merged):,}")
print(f"Points after masking:  {len(df_masked):,}")

`filter_dataframe` operates on a `pandas.DataFrame` already loaded in memory. Use `PointFileIO.read` to load the data beforehand and `PointFileIO.write` to export the filtered result.

In [ ]:
masked_path = "../data/gis/processed/bat_sai_10m-50m_9377_masked.xyz"
PointFileIO(masked_path).write(df_masked)

## Building a profile axis

`ProfileAxis` constructs the 1D sampling axis of a straight-line profile. It supports constant or variable spacing and can be built either from planar coordinates or from a nominal profile length. The axis is the input required by `ProfileInterpolator` to sample Z values from a point cloud.

In [ ]:
start = (4054860.17, 2962441.64)  # replace with your coordinates
end   = (4051814.07, 2963515.97)  # replace with your coordinates

axis = ProfileAxis.from_coordinates(
    start=start,
    end=end,
    dx=10.0,
    crs="EPSG:9377", # optional
)

Once built, the axis exposes three properties. `distance_axis` returns the cumulative distances from the origin. `coordinates` returns the planar coordinates of each sampling point. `adjusted_end` returns the recalculated endpoint after spacing adjustment, which may differ slightly from the nominal `end` passed at construction.

In [ ]:
axis.distance_axis.head()

In [ ]:
axis.coordinates.head()

In [ ]:
axis.adjusted_end

When planar coordinates are not available, `ProfileAxis.from_length` builds the axis from a nominal total length. The properties `coordinates` and `adjusted_end` are not available in this mode.

In [ ]:
axis_from_length = ProfileAxis.from_length(length=500.0, dx=10.0)
axis_from_length.distance_axis.head()

Variable spacing is supported by passing a `dict` as `dx`, each key defines the absolute distance from the origin up to which the associated spacing is used. This option is also available in `from_coordinates`.

In [ ]:
# Variable spacing with a dict: dx=5 up to 200m, dx=10 up to 500m
axis_dict = ProfileAxis.from_length(
    length=500.0,
    dx={200: 5.0, 500: 10.0},
)
axis_dict.distance_axis.head()

## Building a grid

`Grid` constructs the 2D sampling axes of a regular grid. It supports constant spacing and can be built either from planar coordinates or from nominal grid extents read from a shapefile.

In [ ]:
grid = Grid.from_shapefile(
    shapefile_path="../data/gis/raw/small_domain.shp",dx = 10, dy = 10
)

Once built, the `Grid` class provides properties to access the grid coordinates as 2D DataFrames. Both absolute and relative coordinates are available. Absolute coordinates are the original planar coordinates of the grid points, while relative coordinates are calculated by subtracting the starting point from each coordinate, effectively translating the grid to a local reference frame.

In [ ]:
grid.absolute_x_coordinates.head()

In [ ]:
grid.absolute_y_coordinates.head()

The number of grid points in each direction can be easily accessed via the `nx` and `ny` properties, which return the length of the corresponding 1D coordinate arrays.

In [ ]:
grid.nx, grid.ny

## Interpolating a bathymetric profile

`ProfileInterpolator` samples Z values from a scattered XYZ point cloud at the positions defined by a `ProfileAxis`. It requires an axis built with `from_coordinates` since planar coordinates are needed to locate the profile within the point cloud.

The point cloud must be loaded beforehand using `PointFileIO`. Loading it as a `GeoDataFrame` via `read_as_geodataframe` enables CRS consistency checking between the profile axis and the point cloud.

In [ ]:
xyz_path = "../data/gis/processed/bat_sai_10m-50m_9377_merged.xyz"
gdf_xyz = PointFileIO(xyz_path, crs=9377).read_as_geodataframe()

Before interpolating, define the profile axis using planar coordinates within the XYZ domain. The CRS must match the point cloud.

In [ ]:
start = (4054860.17, 2962441.64)  # replace with your coordinates
end   = (4051814.07, 2963515.97)  # replace with your coordinates

axis = ProfileAxis.from_coordinates(
    start=start,
    end=end,
    dx=10.0,
    crs="EPSG:9377",
)

In [ ]:
interpolator = ProfileInterpolator(
    axis=axis,
    xyz=gdf_xyz,
)

`ProfileInterpolator` exposes two output DataFrames. `profile_sz` returns the along-profile distance and the interpolated Z value at each sampling point. `profile_xyz` returns the planar coordinates and Z, following the column convention of `PointFileIO` so it can be exported directly.

In [ ]:
interpolator.profile_sz.head()

In [ ]:
interpolator.profile_xyz.head()

The `profile_xyz` DataFrame can be exported directly using `PointFileIO` without any transformation.

In [ ]:
profile_path = "../data/gis/processed/profile_xyz.xyz"

PointFileIO(profile_path).write(interpolator.profile_xyz)